# TC_04: FCNN with fault value perturbation
Inject fault values at 5%, 10%, 20% levels. Run full diagnostic pipeline at each level and compare against TC_03 (Golden Baseline)

In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.append('..')
os.chdir('..')
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

from src.utils.config_loader import load_config
from src.utils.data_loader import load_tabular_data
from src.data_diagnostics.quality_checks import check_tabular_quality
from src.utils.visualization import plot_class_distribution, plot_correlation, plot_feature_boxplots
from src.utils.performance_tracker import PerformanceTracker
from src.models.tabular_models import train_fcnn, evaluate_model_fcnn
from src.inference_diagnostics.uncertainty import deep_ensemble_prediction_tab, mc_dropout_prediction_tab
from src.inference_diagnostics.explainability import shap_tab, lime_tab,calculate_consistency_tabular
from src.inference_diagnostics.flagging import assign_flags,evaluate_flags
from src.utils.visualization import plot_uncertainty_distribution, plot_flag_distribution
from src.utils.report_generator import generate_report,save_report
from src.data_diagnostics.perturbations import inject_fault_value_tab


config = load_config()
tracker = PerformanceTracker()

### Load clean data


In [2]:
X_train, X_test, y_train, y_test = load_tabular_data(config)
quality_report = check_tabular_quality(X_train, y_train, config)

# Load seed
seed = config['random_seeds']['primary_seed']
fractions =  config['stage1_data_quality']['perturbations']['missing_values']['fractions']

Loading dataset.
No missing value found.
Data loaded for 202944 train, 50736 test
 Features: 21
EDA started for tabular data.
Samples: 202944, Features: 21
Class distribution: {0: 174667, 1: 28277}
Imbalance ratio: 0.1619
Duplicate rows: 18580
Total outlier count: 89295
EDA completed for Diabetes Health Indicators


### Run full pipeline for each perturbation levels

In [3]:
all_results = {}
# Get golden baseling MC Dropout and Deep Ensemble threshold values
mc_threshold = config['stage3_inference']['flagging']['mc_dropout_uncertainty_threshold']
de_threshold = config['stage3_inference']['flagging']['deep_ensemble_uncertainty_threshold']

for fraction in fractions:
    print(f"Inject fault values for fraction: {fraction * 100:.0f}%")

    tracker = PerformanceTracker()

    # Inject fault values
    X_train_corrupted = inject_fault_value_tab(X_train, fraction, seed)
    corrupted_report = check_tabular_quality(X_train_corrupted, y_train, config)
    plot_class_distribution(corrupted_report, f"FCNN Diabetes fault values fraction: {fraction * 100:.0f}% ", config)
    plot_correlation(corrupted_report, f"FCNN Diabetes fault values fraction: {fraction * 100:.0f}%", config)
    plot_feature_boxplots(X_train_corrupted, f"FCNN Diabetes fault values fraction: {fraction * 100:.0f}%", config)

    # Train FCNN with fault values
    tracker.start_performance_track()
    fcnn_model = train_fcnn(X_train_corrupted, y_train, config)
    tracker.stop_performance_track(f"FCNN training fault values fraction: {fraction * 100:.0f}%")

    tracker.start_performance_track()
    fcnn_accuracy, fcnn_prediction, fcnn_report = evaluate_model_fcnn(fcnn_model, X_test, y_test, config)
    tracker.stop_performance_track(f"FCNN evaluation fault values fraction: {fraction * 100:.0f}%")

    # MC Dropout
    tracker.start_performance_track()
    mc_means_probabilities, mc_uncertainty = mc_dropout_prediction_tab(fcnn_model, X_test, config)
    tracker.stop_performance_track(f"FCNN MC Dropout fault values fraction: {fraction * 100:.0f}%")

    print(f"MC Dropout uncertainty status:")
    print(f"Mean: {mc_uncertainty.mean()}")
    print(f"Max: {mc_uncertainty.max()}")
    print(f" 90th percentile (threshold): {mc_threshold}")

    plot_uncertainty_distribution(mc_uncertainty, f"FCNN MC Dropout fault values fraction: {fraction * 100:.0f}% ", mc_threshold, config)

    # Deep Ensemble
    tracker.start_performance_track()
    de_means_probabilities, de_uncertainty = deep_ensemble_prediction_tab(train_fcnn, X_train_corrupted, y_train, X_test, config)
    tracker.stop_performance_track(f"FCNN Deep Ensemble fault values fraction: {fraction * 100:.0f}%")

    print(f"Deep Ensembl uncertainty status:")
    print(f"Mean: {de_uncertainty.mean()}")
    print(f"Max: {de_uncertainty.max()}")
    print(f" 90th percentile (threshold): {de_threshold}")

    plot_uncertainty_distribution(de_uncertainty, f"FCNN Deep Ensemble fault values fraction: {fraction * 100:.0f}%", de_threshold, config)

    # SHAP
    tracker.start_performance_track()
    shap_values, shap_samples = shap_tab(fcnn_model, X_train_corrupted, X_test, config, is_pytorch = True)
    tracker.stop_performance_track(f"FCNN SHAP fault values fraction: {fraction * 100:.0f}%")
    print(f"SHAP values shape: {shap_values.shape}")

    # LIME
    tracker.start_performance_track()
    lime_explanations, lime_samples = lime_tab(fcnn_model, X_train_corrupted, X_test, config, is_pytorch = True)
    tracker.stop_performance_track(f"FCNN LIME fault values fraction: {fraction * 100:.0f}%")
    print(f"LIME explanations: {len(lime_explanations)}")

    # Calculate consistency
    feature_names = list(X_train.columns)
    consistency_scores = calculate_consistency_tabular(shap_values, lime_explanations, feature_names, top=5)

    # MC Dropout with XAI flags
    mc_y_predictions = mc_means_probabilities[:len(consistency_scores)].argmax(axis = 1)
    mc_flags = assign_flags(mc_uncertainty[:len(consistency_scores)], consistency_scores, mc_threshold, 0.5)
    mc_flag_results = evaluate_flags(mc_flags, mc_y_predictions, y_test[:len(consistency_scores)])
    plot_flag_distribution(mc_flags, f"FCNN MC Dropout + XAI Flagging fault values fraction: {fraction * 100:.0f}%", config)

    # Deep Ensemble flags
    de_y_predictions = de_means_probabilities[:len(consistency_scores)].argmax(axis = 1)
    de_flags = assign_flags(de_uncertainty[:len(consistency_scores)], consistency_scores, de_threshold, 0.5)
    de_flag_results = evaluate_flags(de_flags, de_y_predictions, y_test[:len(consistency_scores)])
    plot_flag_distribution(de_flags, f"FCNN Deep Ensembles + XAI Flagging fault values fraction: {fraction * 100:.0f}%", config)

    # MC Dropout without XAI (constant consistency)
    con_consistency = np.ones(len(consistency_scores))
    mc_flags_uq_only = assign_flags(mc_uncertainty[:len(consistency_scores)], con_consistency, mc_threshold, 0.5)

    print(f"MC Dropout without XAI Flagging:")
    mc_only_flag_results = evaluate_flags(mc_flags_uq_only, mc_y_predictions, y_test[:len(consistency_scores)])
    plot_flag_distribution(mc_flags_uq_only, f"FCNN MC Dropout only fault values fraction: {fraction * 100:.0f}%", config)

    print(f"MC Dropout with XAI green accuracy: {mc_flag_results['GREEN']['accuracy']}")
    print(f"MC Dropout without XAI green accuracy: {mc_only_flag_results['GREEN']['accuracy']}")
    print(f"Improvement with XAI: {mc_flag_results['GREEN']['accuracy'] - mc_only_flag_results['GREEN']['accuracy']}")

    # Deep Ensemble without XAI (constant consistency)
    de_flags_uq_only = assign_flags(de_uncertainty[:len(consistency_scores)], con_consistency, de_threshold, 0.5)

    print(f"Deep Ensemble without XAI Flagging:")
    de_only_flag_results = evaluate_flags(de_flags_uq_only, de_y_predictions, y_test[:len(consistency_scores)])
    plot_flag_distribution(de_flags_uq_only, f"FCNN Deep Ensembles only fault values fraction: {fraction * 100:.0f}%", config)

    print(f"Deep Ensembles with XAI green accuracy: {de_flag_results['GREEN']['accuracy']}")
    print(f"Deep Ensembles without XAI green accuracy: {de_only_flag_results['GREEN']['accuracy']}")
    print(f"Improvement with XAI: {de_flag_results['GREEN']['accuracy'] - de_only_flag_results['GREEN']['accuracy']}")

    level_result = {
    'perturbation': 'fault values injection',
    'fraction': fraction,
    'model': 'FCNN',
    'accuracy': round(fcnn_accuracy, 4),
    'classification_report': fcnn_report,
    'quality_report': quality_report['feature_stats'],
    'corrupted_quality_report': corrupted_report['feature_stats'],
    'mc_uncertainty': {
        'mean': round(float(mc_uncertainty.mean()), 8),
        'max': round(float(mc_uncertainty.max()), 8),
        'threshold': mc_threshold
    },
    'de_uncertainty': {
        'mean': round(float(de_uncertainty.mean()), 8),
        'max': round(float(de_uncertainty.max()), 8),
        'threshold': de_threshold
    },
    'consistency':{
        'mean': round(float(consistency_scores.mean()), 4),
        'min': round(float(consistency_scores.min()), 4),
        'max': round(float(consistency_scores.max()), 4)
    },
    'flagging_mc': mc_flag_results,
    'flagging_de': de_flag_results,
    'flagging_mc_only': mc_only_flag_results,
    'mc_vs_mc_plus_xai': round(mc_flag_results['GREEN']['accuracy'] - mc_only_flag_results['GREEN']['accuracy'], 4),
    'flagging_de_only': de_only_flag_results,
    'de_vs_de_plus_xai': round(de_flag_results['GREEN']['accuracy'] - de_only_flag_results['GREEN']['accuracy'], 4),
    'performance': tracker.get_results()
    }

    all_results[f"{fraction * 100:.0f}"] = level_result

    # Save individual report
    report_output = generate_report(config,f'Diabetes - FCNN fault values fraction: {fraction * 100:.0f}%', stage1_result = level_result )
    save_report(report_output, f'TC_04_FCNN_Fault_Values_{fraction * 100:.0f}_Fraction.json', config)


Inject fault values for fraction: 5%
Injecting 5% fault values
Perturbed 5 values from the data
EDA started for tabular data.
Samples: 202944, Features: 21
Class distribution: {0: 174667, 1: 28277}
Imbalance ratio: 0.1619
Duplicate rows: 20048
Total outlier count: 100893
EDA completed for Diabetes Health Indicators
FCNN training started.
Epoch 10/50
Epoch 20/50
 Early stopping at epoch 30
FCNN training completed.
FCNN training fault values fraction: 5%: Time:86.01s, Memory:601.17MB
Model evaluation started for FCNN.
{'0': {'precision': 0.8724235675237902, 'recall': 0.9867634598209174, 'f1-score': 0.9260775653631645, 'support': 43667.0}, '1': {'precision': 0.5705794947994056, 'recall': 0.1086433724713538, 'f1-score': 0.18253119429590017, 'support': 7069.0}, 'accuracy': 0.8644157994323557, 'macro avg': {'precision': 0.721501531161598, 'recall': 0.5477034161461356, 'f1-score': 0.5543043798295324, 'support': 50736.0}, 'weighted avg': {'precision': 0.8303679117746443, 'recall': 0.8644157994

  0%|          | 0/200 [00:00<?, ?it/s]

SHAP finished. Explained: 200 samples.
FCNN SHAP fault values fraction: 5%: Time:7.35s, Memory:11.15MB
SHAP values shape: (200, 21, 2)
LIME started.
Explained 50 / 200 samples.
Explained 100 / 200 samples.
Explained 150 / 200 samples.
Explained 200 / 200 samples.
LIME finished. Explained: 200 samples.
FCNN LIME fault values fraction: 5%: Time:1.91s, Memory:0.00MB
LIME explanations: 200
Calculate consistency tabular.
RED: Count: 1 Accuracy:100.0%
YELLOW: Count: 72 Accuracy:81.9%
GREEN: Count: 127 Accuracy:89.8%
Flagging system is not working, needs threshold adjustment
Saved: results/figures\flagging_distribution_fcnn_mc_dropout_+_xai_flagging_fault_values_fraction:_5%.png
RED: Count: 2 Accuracy:100.0%
YELLOW: Count: 69 Accuracy:81.2%
GREEN: Count: 129 Accuracy:89.9%
Flagging system is not working, needs threshold adjustment
Saved: results/figures\flagging_distribution_fcnn_deep_ensembles_+_xai_flagging_fault_values_fraction:_5%.png
MC Dropout without XAI Flagging:
RED: Count: 0 Accurac

  0%|          | 0/200 [00:00<?, ?it/s]

SHAP finished. Explained: 200 samples.
FCNN SHAP fault values fraction: 10%: Time:7.44s, Memory:2.41MB
SHAP values shape: (200, 21, 2)
LIME started.
Explained 50 / 200 samples.
Explained 100 / 200 samples.
Explained 150 / 200 samples.
Explained 200 / 200 samples.
LIME finished. Explained: 200 samples.
FCNN LIME fault values fraction: 10%: Time:3.67s, Memory:8.33MB
LIME explanations: 200
Calculate consistency tabular.
RED: Count: 3 Accuracy:33.3%
YELLOW: Count: 79 Accuracy:81.0%
GREEN: Count: 118 Accuracy:90.7%
Flagging system is working
Saved: results/figures\flagging_distribution_fcnn_mc_dropout_+_xai_flagging_fault_values_fraction:_10%.png
RED: Count: 3 Accuracy:66.7%
YELLOW: Count: 80 Accuracy:82.5%
GREEN: Count: 117 Accuracy:90.6%
Flagging system is working
Saved: results/figures\flagging_distribution_fcnn_deep_ensembles_+_xai_flagging_fault_values_fraction:_10%.png
MC Dropout without XAI Flagging:
RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 17 Accuracy:47.1%
GREEN: Count: 183 Accur

  0%|          | 0/200 [00:00<?, ?it/s]

SHAP finished. Explained: 200 samples.
FCNN SHAP fault values fraction: 20%: Time:7.32s, Memory:0.01MB
SHAP values shape: (200, 21, 2)
LIME started.
Explained 50 / 200 samples.
Explained 100 / 200 samples.
Explained 150 / 200 samples.
Explained 200 / 200 samples.
LIME finished. Explained: 200 samples.
FCNN LIME fault values fraction: 20%: Time:2.38s, Memory:0.00MB
LIME explanations: 200
Calculate consistency tabular.
RED: Count: 7 Accuracy:71.4%
YELLOW: Count: 78 Accuracy:85.9%
GREEN: Count: 115 Accuracy:90.4%
Flagging system is working
Saved: results/figures\flagging_distribution_fcnn_mc_dropout_+_xai_flagging_fault_values_fraction:_20%.png
RED: Count: 12 Accuracy:91.7%
YELLOW: Count: 69 Accuracy:85.5%
GREEN: Count: 119 Accuracy:87.4%
Flagging system is not working, needs threshold adjustment
Saved: results/figures\flagging_distribution_fcnn_deep_ensembles_+_xai_flagging_fault_values_fraction:_20%.png
MC Dropout without XAI Flagging:
RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 25 Accur